## 1.DummyClassifier

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

In [ ]:
# 定位项目路径并创建输出目录
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Cannot locate project root. Please run this notebook inside the project folder or notebook folder."
    )

DATA_DIR = PROJECT_DIR / "data" / "raw"

OUTPUT_DIR = PROJECT_DIR / "reports" / "model" / "baseline" / "dummy"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_submission_path = DATA_DIR / "sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample submission shape:", sample_submission.shape)

display(train.head())

In [ ]:
#划分特征与目标变量
TARGET_COL = "Churn"

if TARGET_COL not in train.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in train.csv")

id_candidates = {
    "id",
    "customerid",
    "customer_id",
    "clientid",
    "client_id",
    "userid",
    "user_id",
}

id_cols = [col for col in train.columns if col.lower() in id_candidates]

feature_cols = [col for col in train.columns if col not in id_cols + [TARGET_COL]]

X = train[feature_cols]
y = train[TARGET_COL]

print("ID columns:", id_cols)
print("Target column:", TARGET_COL)
print("Number of features:", len(feature_cols))
print("Target distribution:")
display(y.value_counts(normalize=True).rename("proportion").to_frame())

In [ ]:
#划分训练集与验证集
RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)

print("Train target distribution:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())

print("Valid target distribution:")
display(y_valid.value_counts(normalize=True).rename("proportion").to_frame())

In [ ]:
#训练基线模型并评估性能
dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=RANDOM_STATE,
)

dummy_model.fit(X_train, y_train)

y_valid_pred = dummy_model.predict(X_valid)

print("Dummy model classes:", dummy_model.classes_)
print("Predicted class distribution:")
display(pd.Series(y_valid_pred).value_counts(normalize=True).rename("proportion").to_frame())

In [ ]:
#计算验证集指标
classes = dummy_model.classes_

if len(classes) != 2:
    raise ValueError("This notebook assumes a binary classification task.")

# 自动识别正类
if 1 in classes:
    positive_class = 1
elif "Yes" in classes:
    positive_class = "Yes"
elif "yes" in classes:
    positive_class = "yes"
elif "True" in classes:
    positive_class = "True"
else:
    positive_class = classes[1]

positive_index = np.where(classes == positive_class)[0][0]

y_valid_binary = (y_valid == positive_class).astype(int)

if hasattr(dummy_model, "predict_proba"):
    y_valid_proba = dummy_model.predict_proba(X_valid)[:, positive_index]
else:
    y_valid_proba = None

metrics = {
    "model": "DummyClassifier",
    "strategy": "most_frequent",
    "positive_class": positive_class,
    "train_size": len(X_train),
    "valid_size": len(X_valid),
    "accuracy": accuracy_score(y_valid, y_valid_pred),
    "balanced_accuracy": balanced_accuracy_score(y_valid, y_valid_pred),
    "precision": precision_score(y_valid, y_valid_pred, pos_label=positive_class, zero_division=0),
    "recall": recall_score(y_valid, y_valid_pred, pos_label=positive_class, zero_division=0),
    "f1": f1_score(y_valid, y_valid_pred, pos_label=positive_class, zero_division=0),
}

if y_valid_proba is not None:
    metrics["roc_auc"] = roc_auc_score(y_valid_binary, y_valid_proba)
    metrics["average_precision"] = average_precision_score(y_valid_binary, y_valid_proba)

metrics_df = pd.DataFrame([metrics])

display(metrics_df)

In [ ]:
#保存分类报告和混淆矩阵
report_dict = classification_report(
    y_valid,
    y_valid_pred,
    output_dict=True,
    zero_division=0,
)

classification_report_df = pd.DataFrame(report_dict).T

cm = confusion_matrix(y_valid, y_valid_pred, labels=classes)
confusion_matrix_df = pd.DataFrame(
    cm,
    index=[f"actual_{cls}" for cls in classes],
    columns=[f"pred_{cls}" for cls in classes],
)

metrics_df.to_csv(TABLE_DIR / "dummy_metrics.csv", index=False)
classification_report_df.to_csv(TABLE_DIR / "dummy_classification_report.csv")
confusion_matrix_df.to_csv(TABLE_DIR / "dummy_confusion_matrix.csv")

display(classification_report_df)
display(confusion_matrix_df)

print("Saved tables to:", TABLE_DIR)

## 2. LogisticRegression Baseline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

In [ ]:
LOGISTIC_OUTPUT_DIR = PROJECT_DIR / "reports" / "model" / "baseline" / "logistic_regression"
LOGISTIC_TABLE_DIR = LOGISTIC_OUTPUT_DIR / "tables"
LOGISTIC_FIGURE_DIR = LOGISTIC_OUTPUT_DIR / "figures"

LOGISTIC_TABLE_DIR.mkdir(parents=True, exist_ok=True)
LOGISTIC_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("LOGISTIC_OUTPUT_DIR:", LOGISTIC_OUTPUT_DIR)

In [ ]:
#识别数值型和类别型特征
numeric_cols = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()

print("Number of numeric columns:", len(numeric_cols))
print("Numeric columns:", numeric_cols)

print("Number of categorical columns:", len(categorical_cols))
print("Categorical columns:", categorical_cols)

In [ ]:
#构建预处理器和LogisticRegression模型
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                solver="lbfgs",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_model

In [ ]:
#训练LogisticRegression模型
logistic_model.fit(X_train, y_train)

print("LogisticRegression training completed.")

In [ ]:
#验证集预测
y_valid_pred_logistic = logistic_model.predict(X_valid)

classes = logistic_model.named_steps["model"].classes_

if 1 in classes:
    positive_class = 1
elif "Yes" in classes:
    positive_class = "Yes"
elif "yes" in classes:
    positive_class = "yes"
elif "True" in classes:
    positive_class = "True"
else:
    positive_class = classes[1]

positive_index = np.where(classes == positive_class)[0][0]

y_valid_proba_logistic = logistic_model.predict_proba(X_valid)[:, positive_index]
y_valid_binary = (y_valid == positive_class).astype(int)

print("Model classes:", classes)
print("Positive class:", positive_class)

print("Predicted class distribution:")
display(
    pd.Series(y_valid_pred_logistic)
    .value_counts(normalize=True)
    .rename("proportion")
    .to_frame()
)

In [ ]:
# 计算逻辑回归模型的评估指标
logistic_metrics = {
    "model": "LogisticRegression",
    "positive_class": positive_class,
    "train_size": len(X_train),
    "valid_size": len(X_valid),
    "accuracy": accuracy_score(y_valid, y_valid_pred_logistic),
    "balanced_accuracy": balanced_accuracy_score(y_valid, y_valid_pred_logistic),
    "precision": precision_score(
        y_valid,
        y_valid_pred_logistic,
        pos_label=positive_class,
        zero_division=0,
    ),
    "recall": recall_score(
        y_valid,
        y_valid_pred_logistic,
        pos_label=positive_class,
        zero_division=0,
    ),
    "f1": f1_score(
        y_valid,
        y_valid_pred_logistic,
        pos_label=positive_class,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(y_valid_binary, y_valid_proba_logistic),
    "average_precision": average_precision_score(y_valid_binary, y_valid_proba_logistic),
}

logistic_metrics_df = pd.DataFrame([logistic_metrics])

display(logistic_metrics_df)

In [ ]:
#保存LogisticRegression分类报告和混淆矩阵
logistic_report_dict = classification_report(
    y_valid,
    y_valid_pred_logistic,
    output_dict=True,
    zero_division=0,
)

logistic_classification_report_df = pd.DataFrame(logistic_report_dict).T

logistic_cm = confusion_matrix(
    y_valid,
    y_valid_pred_logistic,
    labels=classes,
)

logistic_confusion_matrix_df = pd.DataFrame(
    logistic_cm,
    index=[f"actual_{cls}" for cls in classes],
    columns=[f"pred_{cls}" for cls in classes],
)

logistic_metrics_df.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_metrics.csv",
    index=False,
)

logistic_classification_report_df.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_classification_report.csv",
    index_label="class",
)

logistic_confusion_matrix_df.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_confusion_matrix.csv",
)

display(logistic_classification_report_df)
display(logistic_confusion_matrix_df)

print("Saved LogisticRegression tables to:", LOGISTIC_TABLE_DIR)

In [ ]:
#提取LogisticRegression模型的特征重要性（系数）并保存
fitted_preprocessor = logistic_model.named_steps["preprocessor"]
fitted_logistic = logistic_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = fitted_logistic.coef_[0]

coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coefficients,
        "abs_coefficient": np.abs(coefficients),
    }
)

coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

coef_df.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_feature_coefficients.csv",
    index=False,
)

display(coef_df.head(30))

In [ ]:
#找到最重要的正向和负向特征
top_positive_coef = coef_df.sort_values("coefficient", ascending=False).head(20)
top_negative_coef = coef_df.sort_values("coefficient", ascending=True).head(20)

top_positive_coef.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_top_positive_coefficients.csv",
    index=False,
)

top_negative_coef.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_top_negative_coefficients.csv",
    index=False,
)

print("Top positive coefficients:")
display(top_positive_coef)

print("Top negative coefficients:")
display(top_negative_coef)

## 3. RandomForest Baseline

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
RF_OUTPUT_DIR = PROJECT_DIR / "reports" / "model" / "baseline" / "random_forest"
RF_TABLE_DIR = RF_OUTPUT_DIR / "tables"
RF_FIGURE_DIR = RF_OUTPUT_DIR / "figures"

RF_TABLE_DIR.mkdir(parents=True, exist_ok=True)
RF_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("RF_OUTPUT_DIR:", RF_OUTPUT_DIR)

In [ ]:
#构建RandomForest预处理器
rf_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

rf_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", rf_numeric_transformer, numeric_cols),
        ("cat", rf_categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

In [ ]:
#构建RandomForest模型
rf_model = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                class_weight=None,
            ),
        ),
    ]
)

rf_model

In [ ]:
#训练RandomForest模型
rf_model.fit(X_train, y_train)

print("RandomForest training completed.")

In [ ]:
#预测验证集上的结果
y_valid_pred_rf = rf_model.predict(X_valid)

rf_classes = rf_model.named_steps["model"].classes_

if 1 in rf_classes:
    rf_positive_class = 1
elif "Yes" in rf_classes:
    rf_positive_class = "Yes"
elif "yes" in rf_classes:
    rf_positive_class = "yes"
elif "True" in rf_classes:
    rf_positive_class = "True"
else:
    rf_positive_class = rf_classes[1]

rf_positive_index = np.where(rf_classes == rf_positive_class)[0][0]

y_valid_proba_rf = rf_model.predict_proba(X_valid)[:, rf_positive_index]
y_valid_binary_rf = (y_valid == rf_positive_class).astype(int)

print("Model classes:", rf_classes)
print("Positive class:", rf_positive_class)

print("Predicted class distribution:")
display(
    pd.Series(y_valid_pred_rf)
    .value_counts(normalize=True)
    .rename("proportion")
    .to_frame()
)

In [ ]:
#计算RandomForest模型的评估指标
rf_metrics = {
    "model": "RandomForest",
    "positive_class": rf_positive_class,
    "train_size": len(X_train),
    "valid_size": len(X_valid),
    "accuracy": accuracy_score(y_valid, y_valid_pred_rf),
    "balanced_accuracy": balanced_accuracy_score(y_valid, y_valid_pred_rf),
    "precision": precision_score(
        y_valid,
        y_valid_pred_rf,
        pos_label=rf_positive_class,
        zero_division=0,
    ),
    "recall": recall_score(
        y_valid,
        y_valid_pred_rf,
        pos_label=rf_positive_class,
        zero_division=0,
    ),
    "f1": f1_score(
        y_valid,
        y_valid_pred_rf,
        pos_label=rf_positive_class,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(y_valid_binary_rf, y_valid_proba_rf),
    "average_precision": average_precision_score(y_valid_binary_rf, y_valid_proba_rf),
}

rf_metrics_df = pd.DataFrame([rf_metrics])

display(rf_metrics_df)

In [ ]:
#保存RandomForest分类报告和混淆矩阵
rf_report_dict = classification_report(
    y_valid,
    y_valid_pred_rf,
    output_dict=True,
    zero_division=0,
)

rf_classification_report_df = pd.DataFrame(rf_report_dict).T

rf_cm = confusion_matrix(
    y_valid,
    y_valid_pred_rf,
    labels=rf_classes,
)

rf_confusion_matrix_df = pd.DataFrame(
    rf_cm,
    index=[f"actual_{cls}" for cls in rf_classes],
    columns=[f"pred_{cls}" for cls in rf_classes],
)

rf_metrics_df.to_csv(
    RF_TABLE_DIR / "random_forest_metrics.csv",
    index=False,
)

rf_classification_report_df.to_csv(
    RF_TABLE_DIR / "random_forest_classification_report.csv",
    index_label="class",
)

rf_confusion_matrix_df.to_csv(
    RF_TABLE_DIR / "random_forest_confusion_matrix.csv",
)

display(rf_classification_report_df)
display(rf_confusion_matrix_df)

print("Saved RandomForest tables to:", RF_TABLE_DIR)

In [ ]:
#保存RandomForest特征重要性
rf_fitted_preprocessor = rf_model.named_steps["preprocessor"]
rf_fitted_model = rf_model.named_steps["model"]

rf_feature_names = rf_fitted_preprocessor.get_feature_names_out()
rf_feature_importances = rf_fitted_model.feature_importances_

rf_feature_importance_df = pd.DataFrame(
    {
        "feature": rf_feature_names,
        "importance": rf_feature_importances,
    }
)

rf_feature_importance_df = rf_feature_importance_df.sort_values(
    "importance",
    ascending=False,
)

rf_feature_importance_df.to_csv(
    RF_TABLE_DIR / "random_forest_feature_importance.csv",
    index=False,
)

display(rf_feature_importance_df.head(30))

In [ ]:
#聚合One-Hot后的原始变量重要性
def clean_original_feature_name(feature_name: str) -> str:
    """
    Convert transformed feature name back to original feature group.

    Examples
    --------
    num__tenure -> tenure
    cat__Contract_Month-to-month -> Contract
    cat__PaymentMethod_Electronic check -> PaymentMethod
    """
    feature_name = str(feature_name)

    if feature_name.startswith("num__"):
        return feature_name.replace("num__", "", 1)

    if feature_name.startswith("cat__"):
        name = feature_name.replace("cat__", "", 1)

        matched_cols = [
            col for col in categorical_cols
            if name == col or name.startswith(f"{col}_")
        ]

        if matched_cols:
            return sorted(matched_cols, key=len, reverse=True)[0]

        return name

    return feature_name


rf_feature_importance_df["original_feature"] = rf_feature_importance_df["feature"].apply(
    clean_original_feature_name
)

rf_original_feature_importance_df = (
    rf_feature_importance_df
    .groupby("original_feature", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

rf_original_feature_importance_df.to_csv(
    RF_TABLE_DIR / "random_forest_original_feature_importance.csv",
    index=False,
)

display(rf_original_feature_importance_df)

In [ ]:
#比较三个模型的性能指标
metrics_files = [
    PROJECT_DIR / "reports" / "model" / "baseline" / "dummy" / "tables" / "dummy_metrics.csv",
    PROJECT_DIR / "reports" / "model" / "baseline" / "logistic_regression" / "tables" / "logistic_metrics.csv",
    RF_TABLE_DIR / "random_forest_metrics.csv",
]

metrics_dfs = []

for path in metrics_files:
    if path.exists():
        temp_df = pd.read_csv(path)
        metrics_dfs.append(temp_df)
    else:
        print("Missing metrics file:", path)

baseline_metrics_comparison_df = pd.concat(metrics_dfs, ignore_index=True)

baseline_metrics_comparison_df.to_csv(
    RF_TABLE_DIR / "baseline_metrics_comparison.csv",
    index=False,
)

display(baseline_metrics_comparison_df)

In [ ]:
# ============================================================
# LightGBM Baseline Model
# 作用：
# 1. 构建LightGBM基准模型
# 2. 自动完成数值/类别特征预处理
# 3. 输出验证集指标、分类报告、混淆矩阵
# 4. 输出LightGBM特征重要性
# 5. 汇总Dummy、LogisticRegression、RandomForest、LightGBM结果
# 6. 自动生成lightgbm_model_report.md
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

# ------------------------------------------------------------
# 1.检查LightGBM是否已安装
# 如果这里报错，先在VS Code终端运行：
# .\.venv\Scripts\python.exe -m pip install lightgbm
# ------------------------------------------------------------
try:
    from lightgbm import LGBMClassifier
except ImportError as e:
    raise ImportError(
        "当前环境没有安装lightgbm。请先在VS Code终端运行："
        ".\\.venv\\Scripts\\python.exe -m pip install lightgbm"
    ) from e


# ------------------------------------------------------------
# 2.创建LightGBM输出目录
# 所有结果会保存到 reports/model/baseline/lightgbm/
# ------------------------------------------------------------
LGBM_OUTPUT_DIR = PROJECT_DIR / "reports" / "model" / "baseline" / "lightgbm"
LGBM_TABLE_DIR = LGBM_OUTPUT_DIR / "tables"
LGBM_FIGURE_DIR = LGBM_OUTPUT_DIR / "figures"

LGBM_TABLE_DIR.mkdir(parents=True, exist_ok=True)
LGBM_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("LGBM_OUTPUT_DIR:", LGBM_OUTPUT_DIR)


# ------------------------------------------------------------
# 3.重新识别数值变量和类别变量
# 这里使用X_train，保证只基于训练集确定特征类型，避免数据泄漏
# ------------------------------------------------------------
numeric_cols = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

print("数值变量数量:", len(numeric_cols))
print("数值变量:", numeric_cols)
print("类别变量数量:", len(categorical_cols))
print("类别变量:", categorical_cols)


# ------------------------------------------------------------
# 4.构建预处理器
# 数值变量：用中位数填补缺失值
# 类别变量：用众数填补缺失值，然后One-Hot编码
# 注意：LightGBM本身可以处理类别特征，但这里为了和前面模型保持流程一致，
#      仍然使用One-Hot编码，便于与LogisticRegression和RandomForest公平比较
# ------------------------------------------------------------
try:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    # 兼容旧版本scikit-learn
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)

lgbm_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

lgbm_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot_encoder),
    ]
)

lgbm_preprocessor = ColumnTransformer(
    transformers=[
        ("num", lgbm_numeric_transformer, numeric_cols),
        ("cat", lgbm_categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)


# ------------------------------------------------------------
# 5.构建LightGBM模型
# 这是Baseline配置，不做复杂调参
# n_estimators：树的数量
# learning_rate：学习率
# num_leaves：叶子节点数量，控制模型复杂度
# subsample/colsample_bytree：行采样和列采样，减少过拟合
# random_state：保证结果可复现
# ------------------------------------------------------------
lgbm_model = Pipeline(
    steps=[
        ("preprocessor", lgbm_preprocessor),
        (
            "model",
            LGBMClassifier(
                objective="binary",
                n_estimators=1000,
                learning_rate=0.03,
                num_leaves=31,
                max_depth=-1,
                min_child_samples=30,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_alpha=0.0,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            ),
        ),
    ]
)

print("LightGBM模型结构:")
display(lgbm_model)


# ------------------------------------------------------------
# 6.训练LightGBM模型
# ------------------------------------------------------------
lgbm_model.fit(X_train, y_train)

print("LightGBM training completed.")


# ------------------------------------------------------------
# 7.验证集预测
# y_valid_pred_lgbm：类别预测结果
# y_valid_proba_lgbm：预测为正类的概率，用于ROC-AUC和AP
# ------------------------------------------------------------
y_valid_pred_lgbm = lgbm_model.predict(X_valid)

lgbm_classes = lgbm_model.named_steps["model"].classes_

# 自动识别正类
if 1 in lgbm_classes:
    lgbm_positive_class = 1
elif "Yes" in lgbm_classes:
    lgbm_positive_class = "Yes"
elif "yes" in lgbm_classes:
    lgbm_positive_class = "yes"
elif "True" in lgbm_classes:
    lgbm_positive_class = "True"
else:
    lgbm_positive_class = lgbm_classes[1]

lgbm_positive_index = np.where(lgbm_classes == lgbm_positive_class)[0][0]

y_valid_proba_lgbm = lgbm_model.predict_proba(X_valid)[:, lgbm_positive_index]
y_valid_binary_lgbm = (y_valid == lgbm_positive_class).astype(int)

print("模型类别:", lgbm_classes)
print("正类:", lgbm_positive_class)

print("验证集预测类别分布:")
display(
    pd.Series(y_valid_pred_lgbm)
    .value_counts(normalize=True)
    .rename("proportion")
    .to_frame()
)


# ------------------------------------------------------------
# 8.计算验证集评价指标
# Accuracy：总体预测正确率
# Balanced Accuracy：平衡准确率，更适合类别不平衡任务
# Precision：预测为流失的客户中，有多少是真的流失
# Recall：真实流失客户中，有多少被模型识别出来
# F1：Precision和Recall的综合指标
# ROC-AUC：整体排序区分能力
# Average Precision：PR曲线下面积，更关注正类识别
# ------------------------------------------------------------
lgbm_metrics = {
    "model": "LightGBM",
    "positive_class": lgbm_positive_class,
    "train_size": len(X_train),
    "valid_size": len(X_valid),
    "accuracy": accuracy_score(y_valid, y_valid_pred_lgbm),
    "balanced_accuracy": balanced_accuracy_score(y_valid, y_valid_pred_lgbm),
    "precision": precision_score(
        y_valid,
        y_valid_pred_lgbm,
        pos_label=lgbm_positive_class,
        zero_division=0,
    ),
    "recall": recall_score(
        y_valid,
        y_valid_pred_lgbm,
        pos_label=lgbm_positive_class,
        zero_division=0,
    ),
    "f1": f1_score(
        y_valid,
        y_valid_pred_lgbm,
        pos_label=lgbm_positive_class,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(y_valid_binary_lgbm, y_valid_proba_lgbm),
    "average_precision": average_precision_score(y_valid_binary_lgbm, y_valid_proba_lgbm),
}

lgbm_metrics_df = pd.DataFrame([lgbm_metrics])

print("LightGBM验证集指标:")
display(lgbm_metrics_df)


# ------------------------------------------------------------
# 9.保存分类报告和混淆矩阵
# 分类报告：分别查看No类和Yes类的Precision、Recall、F1
# 混淆矩阵：查看TP、FP、TN、FN数量
# ------------------------------------------------------------
lgbm_report_dict = classification_report(
    y_valid,
    y_valid_pred_lgbm,
    output_dict=True,
    zero_division=0,
)

lgbm_classification_report_df = pd.DataFrame(lgbm_report_dict).T

lgbm_cm = confusion_matrix(
    y_valid,
    y_valid_pred_lgbm,
    labels=lgbm_classes,
)

lgbm_confusion_matrix_df = pd.DataFrame(
    lgbm_cm,
    index=[f"actual_{cls}" for cls in lgbm_classes],
    columns=[f"pred_{cls}" for cls in lgbm_classes],
)

lgbm_metrics_df.to_csv(
    LGBM_TABLE_DIR / "lightgbm_metrics.csv",
    index=False,
)

lgbm_classification_report_df.to_csv(
    LGBM_TABLE_DIR / "lightgbm_classification_report.csv",
    index_label="class",
)

lgbm_confusion_matrix_df.to_csv(
    LGBM_TABLE_DIR / "lightgbm_confusion_matrix.csv",
    index_label="actual_class",
)

print("LightGBM分类报告:")
display(lgbm_classification_report_df)

print("LightGBM混淆矩阵:")
display(lgbm_confusion_matrix_df)


# ------------------------------------------------------------
# 10.提取LightGBM特征重要性
# importance_type默认是split，即特征被用于分裂的次数
# 注意：One-Hot后类别变量会被拆开，所以后面还要按原始变量聚合
# ------------------------------------------------------------
lgbm_fitted_preprocessor = lgbm_model.named_steps["preprocessor"]
lgbm_fitted_model = lgbm_model.named_steps["model"]

lgbm_feature_names = lgbm_fitted_preprocessor.get_feature_names_out()
lgbm_feature_importances = lgbm_fitted_model.feature_importances_

lgbm_feature_importance_df = pd.DataFrame(
    {
        "feature": lgbm_feature_names,
        "importance": lgbm_feature_importances,
    }
)

lgbm_feature_importance_df = lgbm_feature_importance_df.sort_values(
    "importance",
    ascending=False,
)

lgbm_feature_importance_df.to_csv(
    LGBM_TABLE_DIR / "lightgbm_feature_importance.csv",
    index=False,
)

print("LightGBM One-Hot后特征重要性Top30:")
display(lgbm_feature_importance_df.head(30))


# ------------------------------------------------------------
# 11.将One-Hot后的特征重要性聚合回原始变量
# 例如：
# cat__Contract_Month-to-month -> Contract
# cat__PaymentMethod_Electronic check -> PaymentMethod
# num__tenure -> tenure
# ------------------------------------------------------------
def clean_original_feature_name(feature_name: str) -> str:
    feature_name = str(feature_name)

    if feature_name.startswith("num__"):
        return feature_name.replace("num__", "", 1)

    if feature_name.startswith("cat__"):
        name = feature_name.replace("cat__", "", 1)

        matched_cols = [
            col for col in categorical_cols
            if name == col or name.startswith(f"{col}_")
        ]

        if matched_cols:
            return sorted(matched_cols, key=len, reverse=True)[0]

        return name

    return feature_name


lgbm_feature_importance_df["original_feature"] = lgbm_feature_importance_df[
    "feature"
].apply(clean_original_feature_name)

lgbm_original_feature_importance_df = (
    lgbm_feature_importance_df
    .groupby("original_feature", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

lgbm_original_feature_importance_df.to_csv(
    LGBM_TABLE_DIR / "lightgbm_original_feature_importance.csv",
    index=False,
)

print("LightGBM按原始变量聚合后的特征重要性:")
display(lgbm_original_feature_importance_df)


# ------------------------------------------------------------
# 12.合并Dummy、LogisticRegression、RandomForest、LightGBM指标
# 方便横向比较所有Baseline模型
# ------------------------------------------------------------
metrics_files = [
    PROJECT_DIR / "reports" / "model" / "baseline" / "dummy" / "tables" / "dummy_metrics.csv",
    PROJECT_DIR / "reports" / "model" / "baseline" / "logistic_regression" / "tables" / "logistic_metrics.csv",
    PROJECT_DIR / "reports" / "model" / "baseline" / "random_forest" / "tables" / "random_forest_metrics.csv",
    LGBM_TABLE_DIR / "lightgbm_metrics.csv",
]

metrics_dfs = []

for path in metrics_files:
    if path.exists():
        temp_df = pd.read_csv(path)
        metrics_dfs.append(temp_df)
    else:
        print("未找到指标文件:", path)

baseline_metrics_comparison_df = pd.concat(metrics_dfs, ignore_index=True)

baseline_metrics_comparison_df.to_csv(
    LGBM_TABLE_DIR / "baseline_metrics_comparison.csv",
    index=False,
)

print("Baseline模型指标对比:")
display(baseline_metrics_comparison_df)


# ------------------------------------------------------------
# 13.生成LightGBM Markdown报告
# 报告会自动写入 lightgbm_model_report.md
# ------------------------------------------------------------
top_lgbm_features = lgbm_original_feature_importance_df.head(20)

lgbm_report_lines = [
    "# LightGBM Baseline Report",
    "",
    "## 1. Model Setting",
    "",
    "- Model: LightGBMClassifier",
    "- Preprocessing for numeric variables: median imputation",
    "- Preprocessing for categorical variables: most frequent imputation and One-Hot encoding",
    "- n_estimators: 1000",
    "- learning_rate: 0.03",
    "- num_leaves: 31",
    "- max_depth: -1",
    "- min_child_samples: 30",
    "- subsample: 0.9",
    "- colsample_bytree: 0.9",
    "- reg_lambda: 1.0",
    f"- Train size: {len(X_train)}",
    f"- Validation size: {len(X_valid)}",
    f"- Positive class: {lgbm_positive_class}",
    "",
    "## 2. Validation Metrics",
    "",
    lgbm_metrics_df.to_markdown(index=False),
    "",
    "## 3. Baseline Model Comparison",
    "",
    baseline_metrics_comparison_df.to_markdown(index=False),
    "",
    "## 4. Top Original Feature Importances",
    "",
    top_lgbm_features.to_markdown(index=False),
    "",
    "## 5. Interpretation",
    "",
    "LightGBM是本项目Baseline阶段的梯度提升树模型。",
    "与RandomForest相比，LightGBM采用Boosting思想逐步修正前一轮模型的误差，通常更适合结构化表格数据。",
    "",
    "本节仍采用统一的训练集和验证集划分，并使用与前面模型一致的预处理流程，",
    "从而保证DummyClassifier、LogisticRegression、RandomForest和LightGBM之间具有可比性。",
    "",
    "评价时不应只关注Accuracy，还应重点比较流失类Recall、F1、Balanced Accuracy、ROC-AUC和Average Precision。",
    "如果LightGBM在这些指标上超过LogisticRegression，说明非线性Boosting模型能够进一步提取客户流失模式。",
    "",
    "特征重要性结果反映变量对模型分裂的贡献。",
    "由于类别变量经过One-Hot编码后会被拆分为多个虚拟变量，因此报告中优先解释按原始变量聚合后的重要性。",
    "需要注意的是，特征重要性只能说明预测贡献，不能直接解释为因果关系。",
]

lgbm_report_text = "\n".join(lgbm_report_lines)

lgbm_report_path = LGBM_OUTPUT_DIR / "lightgbm_model_report.md"
lgbm_report_path.write_text(lgbm_report_text, encoding="utf-8")

print("LightGBM报告已保存到:", lgbm_report_path)
print("LightGBM所有表格已保存到:", LGBM_TABLE_DIR)

In [31]:
# ============================================================
# 生成第一版Kaggle提交文件：LightGBM Baseline Submission
# 作用：
# 1. 使用已经训练好的lgbm_model预测test.csv
# 2. 按sample_submission.csv的格式生成提交文件
# 3. 保存到项目根目录下的submissions文件夹
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1.设置提交文件输出目录
# ------------------------------------------------------------
SUBMISSION_DIR = PROJECT_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

print("SUBMISSION_DIR:", SUBMISSION_DIR)


# ------------------------------------------------------------
# 2.读取测试集和提交模板
# 如果前面已经读过test和sample_submission，这里重复读取也没问题，
# 可以保证当前单元格独立运行时不容易出错
# ------------------------------------------------------------
test_path = DATA_DIR / "test.csv"
sample_submission_path = DATA_DIR / "sample_submission.csv"

test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)

display(sample_submission.head())


# ------------------------------------------------------------
# 3.准备测试集特征
# 注意：
# test没有Churn列，所以只需要去掉ID列，保留和训练集一致的feature_cols
# feature_cols来自前面训练阶段
# ------------------------------------------------------------
X_test = test[feature_cols]

print("X_test shape:", X_test.shape)


# ------------------------------------------------------------
# 4.使用LightGBM预测测试集
# 如果sample_submission中的Churn是0/1或Yes/No类别，就输出类别预测；
# 如果sample_submission中的Churn看起来是概率，就输出正类概率。
# 当前先自动判断。
# ------------------------------------------------------------
test_pred_label = lgbm_model.predict(X_test)

lgbm_classes = lgbm_model.named_steps["model"].classes_

if 1 in lgbm_classes:
    positive_class = 1
elif "Yes" in lgbm_classes:
    positive_class = "Yes"
elif "yes" in lgbm_classes:
    positive_class = "yes"
elif "True" in lgbm_classes:
    positive_class = "True"
else:
    positive_class = lgbm_classes[1]

positive_index = np.where(lgbm_classes == positive_class)[0][0]
test_pred_proba = lgbm_model.predict_proba(X_test)[:, positive_index]

print("模型类别:", lgbm_classes)
print("正类:", positive_class)
print("预测概率范围:", test_pred_proba.min(), "到", test_pred_proba.max())


# ------------------------------------------------------------
# 5.生成提交文件
# 关键：提交文件必须保持sample_submission的列名和列顺序
# ------------------------------------------------------------
submission = sample_submission.copy()

target_col = "Churn"

if target_col not in submission.columns:
    raise ValueError(f"sample_submission中没有找到目标列: {target_col}")

# 自动判断提交格式
# 如果sample_submission的Churn列是浮点数，通常表示需要提交概率
# 如果是整数、字符串或类别，通常表示需要提交类别标签
if pd.api.types.is_float_dtype(submission[target_col]):
    submission[target_col] = test_pred_proba
    submission_type = "probability"
else:
    submission[target_col] = test_pred_label
    submission_type = "label"

print("提交类型:", submission_type)
display(submission.head())


# ------------------------------------------------------------
# 6.保存提交文件
# ------------------------------------------------------------
submission_path = SUBMISSION_DIR / "lightgbm_baseline_submission.csv"
submission.to_csv(submission_path, index=False)

print("提交文件已保存到:")
print(submission_path)


# ------------------------------------------------------------
# 7.检查提交文件基本格式
# ------------------------------------------------------------
print("submission shape:", submission.shape)
print("submission columns:", submission.columns.tolist())

display(submission[target_col].value_counts(dropna=False).head())

SUBMISSION_DIR: d:\A_projects\kaggle-customer-churn-prediction\submissions
test shape: (254655, 20)
sample_submission shape: (254655, 2)


,id,Churn
0,594194,0
1,594195,0
2,594196,0
3,594197,0
4,594198,0


X_test shape: (254655, 19)
模型类别: ['No' 'Yes']
正类: Yes
预测概率范围: 0.00013933723667802273 到 0.986388895421355
提交类型: label


,id,Churn
0,594194,No
1,594195,No
2,594196,No
3,594197,No
4,594198,Yes


提交文件已保存到:
d:\A_projects\kaggle-customer-churn-prediction\submissions\lightgbm_baseline_submission.csv
submission shape: (254655, 2)
submission columns: ['id', 'Churn']


Churn
No     204897
Yes     49758
Name: count, dtype: int64